# Ma Sói — Behavior Cloning trên Colab

Train `Learned Policy V0` từ dataset self-play của bot heuristic (BOT_SELF_LEARNING §17).

**Chuẩn bị ở máy bạn trước** (xem `docs/BOT_SELF_LEARNING_TRAINING.md` Bước 2–4), rồi
nén thành một file và tải lên Google Drive:

```powershell
Compress-Archive -Path ai-training\masoi_training, .tmp\enc-0001 -DestinationPath .tmp\bc-package.zip -Force
```

Zip gồm hai thư mục: `masoi_training/` (3 file Python) và `enc-0001/` (tensor `.bin`).
Không cần tải trajectory JSONL 2,7 GB lên — tensor đã encode nhẹ hơn nhiều và nén rất tốt
vì phần lớn đặc trưng là one-hot.

**Ranh giới không đổi (§39):** luật game và observation encoder ở TypeScript, chạy dưới máy bạn.
Colab chỉ nhận số và train — nó không parse trajectory, không biết luật, nên không có đường nào
để một thông tin ẩn lọt vào tầng train.

Runtime: `Runtime → Change runtime type → T4 GPU`. CPU cũng chạy được (MLP nhỏ), chỉ chậm hơn.

In [ ]:
import torch

print("torch", torch.__version__)
print("cuda ", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Nạp dữ liệu từ Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Sửa cho khớp chỗ bạn đặt file trên Drive.
ZIP = "/content/drive/MyDrive/bc-package.zip"

!rm -rf /content/bc && mkdir -p /content/bc
!unzip -q "$ZIP" -d /content/bc
!ls /content/bc

## 2. Kiểm dữ liệu trước khi train

Ba câu hỏi, hỏi trước khi tiêu một giờ GPU:

1. File có khớp `meta.json` không (một file cụt vẫn reshape "thành công" và làm lệch mọi nhãn đi một hàng).
2. Ba phần train/val/test có cộng lại đúng bằng cả tập không (§15).
3. **Mọi nhãn có phải một nước HỢP LỆ theo chính mask của nó không** — nếu không, dataset đang dạy model đi nước ngoài luật.

In [ ]:
import sys

sys.path.insert(0, "/content/bc")
from masoi_training.data import action_distribution, load

DATA = "/content/bc/enc-0001"
data = load(DATA)  # ném ngay nếu file không khớp meta.json

sizes = {name: len(data.split(name)) for name in ("train", "validation", "test")}
assert sum(sizes.values()) == len(data)
assert data.masks[range(len(data)), data.actions].all(), "có nhãn trỏ vào hành động BẤT HỢP LỆ"
assert set(data.rewards.tolist()) <= {-1.0, 1.0}

print("dataset  ", data.meta.get("datasetVersion"), "commit", data.meta.get("gitCommit"))
print("rows     ", len(data), "| obs", data.obs_size, "| actions", data.action_size)
print("split    ", sizes)
print("lớp hành động (§43):", action_distribution(data))
print("\nOK — train được.")

## 3. Chạy thử 2 epoch

Trước khi chạy thật. Một lỗi cấu hình lộ ra ở đây mất 30 giây, lộ ra ở cell sau mất cả lượt GPU.

In [ ]:
!cd /content/bc && python -m masoi_training.train_bc --data enc-0001 --out /content/smoke --epochs 2

## 4. Train thật

In [ ]:
!cd /content/bc && python -m masoi_training.train_bc \
    --data enc-0001 \
    --out /content/model-v001 \
    --epochs 30 \
    --batch-size 1024 \
    --lr 1e-3 \
    --hidden 128 \
    --seed 12345 \
    --model-id policy-v001

## 5. Đọc kết quả

Số quyết định được là **`metrics.test.agreement`** — tỉ lệ model chọn đúng hành động
mà bot heuristic đã chọn, trên những ván model chưa từng thấy.

Đừng kết luận bằng loss. §17 nói rõ: chưa tái lập được bot thì chưa được sang RL.
Loss giảm mà agreement không tăng nghĩa là model đang học phân bố của lớp đông nhất,
không phải học chơi.

In [ ]:
import json

report = json.load(open("/content/model-v001/metrics.json"))

print("test agreement:", report["metrics"]["test"].get("agreement"))
print("val  agreement:", report["metrics"]["validation"].get("agreement"))
print("\nTheo vai (vai nào model bám kém nhất):")
by_role = report["metrics"]["test"].get("agreementByRole", {})
for role, score in sorted(by_role.items(), key=lambda kv: kv[1]):
    print(f"  {role:<18} {score}")

In [ ]:
import matplotlib.pyplot as plt

history = report["history"]
epochs = [row["epoch"] for row in history]

figure, left = plt.subplots(figsize=(8, 4))
left.plot(epochs, [row["trainLoss"] for row in history], label="train loss")
left.set_xlabel("epoch")
left.set_ylabel("loss")

right = left.twinx()
right.plot(epochs, [row.get("val_agreement") for row in history], color="tab:orange", label="val agreement")
right.set_ylabel("agreement")

figure.legend(loc="upper right")
plt.title("Behavior cloning — loss giảm KHÔNG đủ, agreement mới là thứ phải tăng")
plt.show()

## 6. Lưu model về Drive

`model.onnx` là đường vào runtime TypeScript (§39). `metrics.json` mang `modelId`,
`gitCommit`, `datasetVersion` và `trainingSeed` — một model không truy ngược được về
dataset và commit đã sinh ra nó là một model không tái lập được (§46), nên đừng tách
nó khỏi hai file kia.

In [ ]:
!mkdir -p /content/drive/MyDrive/masoi-models
!cp -r /content/model-v001 /content/drive/MyDrive/masoi-models/
!ls -lh /content/drive/MyDrive/masoi-models/model-v001

## Cách đọc con số

| test agreement | Nghĩa là | Làm gì |
|---|---|---|
| < 0,30 | Chưa học được gì đáng kể | Observation còn nghèo — xem mục giới hạn dưới đây |
| 0,30 – 0,60 | Học được xu hướng, chưa tái lập bot | Tăng `--epochs`/`--hidden`; vẫn chững thì phải làm giàu observation |
| > 0,70 | Tái lập baseline khá tốt | Đủ điều kiện §17 để tính tới RL |

Nếu agreement chững quanh 0,3–0,5, nguyên nhân gần như chắc chắn **không** phải model
quá nhỏ, mà là observation mới chỉ có `suspicion`/`trust` cho mỗi ghế — chưa có
`wolfProbability`, `threat`, `credibility`, `influence` (§8). Chúng nằm trong belief
state của bot nhưng chưa được trace chụp lại. Tăng `--hidden` trong trường hợp đó chỉ
làm model khớp kỹ hơn với một bức tranh thiếu.

Đừng chọn seed cho ra kết quả đẹp nhất. Đổi `--seed` là để kiểm model có ổn định không;
nếu hai seed lệch nhau nhiều thì con số nào cũng chưa kết luận được gì.